# Qwen2.5-0.5B → counter warm-start + recovery finetune (T4)

End-to-end witness for the *finetune-pretrained* initiative: take the fp donor, warm-start it
into the counter format (ternary visible weight + in-state optimizer, no Adam moments / no fp
master / no full grad buffer), then **recover quality by distillation** from the resident fp
teacher. The output is a **PPL-recovery curve** on held-out WikiText:

- `PPL(fp Qwen)` — baseline,
- `PPL(counter, warm-start)` — degraded (ternarization is lossy for a full-precision donor),
- `PPL(counter, distilled)` — recovered, closing the gap to baseline.

Set the runtime to **GPU (T4)**. Everything runs from `scripts/finetune_qwen_counter.py`.

In [ ]:
# 1. Get the repo. It is private -> use a GitHub PAT, or attach it as a Kaggle dataset and skip this.
TOKEN = ""  # <- put a read-scoped PAT here, or leave empty if the repo is already on disk
import os
if not os.path.isdir("memory-native"):
    url = f"https://{TOKEN}@github.com/kharkilirov1/memory-native.git" if TOKEN else "https://github.com/kharkilirov1/memory-native.git"
    !git clone -q --branch main $url
%cd memory-native

In [ ]:
# 2. Install the package + donor extras (transformers/safetensors/accelerate) and the corpus loader.
!pip install -q -e ".[donor]" datasets

In [ ]:
# 3. (optional) tune the run. Defaults are set for a T4 in scripts/finetune_qwen_counter.py.
import scripts.finetune_qwen_counter as ftq
ftq.ROUNDS, ftq.STEPS_PER_ROUND = 8, 50   # distill schedule
ftq.BATCH, ftq.SEQ = 8, 512
ftq.LR = 2e-3
ftq.main()

**Reading the result.** `residual gap` is how much of the warm-start→fp gap is still open after
each round; it should fall toward 0. The final line reports the % of the degradation the distill
closed. If recovery stalls, raise `ROUNDS`/`STEPS_PER_ROUND` or `LR`, or add `CE_ALPHA` weight.
This is the toy-to-real bridge of `tests/test_distill.py` (which proves the mechanism on a tiny
random model); here it runs on the real donor and real text.